In [1]:
from pathlib import Path
import duckdb
import pandas as pd

project_folder = Path(r"D:\electricity-tariff-optimisation")
processed_folder = project_folder / "data" / "processed"

database_file = processed_folder / "electricity_analysis.duckdb"

if not database_file.exists():
    raise FileNotFoundError(f"Database not found: {database_file}")

connection = duckdb.connect(
    str(database_file),
    read_only=True,
)

summary = connection.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT household_id) AS households,
        SUM(energy_kwh) AS total_energy_kwh
    FROM analysis_consumption_2013
""").df()

display(summary.round(3))

print("Customer insights notebook ready.")
print("Existing prepared data loaded; source CSVs were not rescanned.")

,rows,households,total_energy_kwh
0,4344960,248,1038488.042


Customer insights notebook ready.
Existing prepared data loaded; source CSVs were not rescanned.


In [2]:
design_usage = connection.sql("""
    SELECT
        household_id,
        interval_start_utc,
        energy_kwh
    FROM analysis_consumption_2013
    WHERE interval_start_utc < TIMESTAMP '2013-07-01 00:00:00'
""").df()

local_time = (
    pd.to_datetime(design_usage["interval_start_utc"], utc=True)
    .dt.tz_convert("Europe/London")
)

hour = local_time.dt.hour
is_weekday = local_time.dt.dayofweek.lt(5)

design_usage["candidate_band"] = "Shoulder"
design_usage.loc[hour.lt(7), "candidate_band"] = "Off-peak"
design_usage.loc[
    is_weekday & hour.ge(16) & hour.lt(20),
    "candidate_band",
] = "Peak"

bands = ["Peak", "Shoulder", "Off-peak"]

customer_band_kwh = (
    design_usage.groupby(["household_id", "candidate_band"])["energy_kwh"]
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=bands, fill_value=0)
)

customer_features = pd.DataFrame(index=customer_band_kwh.index)

customer_features["design_kwh"] = customer_band_kwh.sum(axis=1)

for band in bands:
    customer_features[f"{band.lower().replace('-', '_')}_share_pct"] = (
        customer_band_kwh[band]
        / customer_features["design_kwh"].replace(0, float("nan"))
        * 100
    )

review_flags = connection.sql("""
    SELECT household_id, review_flag
    FROM selected_customer_review_2013
""").df().set_index("household_id")

customer_features = customer_features.join(
    review_flags,
    validate="one_to_one",
)

assert len(customer_features) == 248
assert customer_features["review_flag"].notna().all()

customer_features.to_csv(
    processed_folder / "customer_design_features_2013.csv"
)

customer_band_kwh.to_csv(
    processed_folder / "customer_design_band_kwh_2013.csv"
)

display(customer_features.head(10).round(2))

print("Customer feature rows:", len(customer_features))
print(
    "Customers with zero total design-period consumption:",
    int(customer_features["design_kwh"].eq(0).sum()),
)

,design_kwh,peak_share_pct,shoulder_share_pct,off_peak_share_pct,review_flag
household_id,,,,,
MAC000002,2187.53,15.15,67.22,17.63,False
MAC000040,5676.66,13.74,64.40,21.86,False
MAC000100,2123.88,17.72,71.73,10.55,False
MAC000145,2338.84,15.68,65.63,18.70,False
MAC000162,1094.85,15.61,64.04,20.36,False
MAC000181,2528.33,19.24,68.14,12.62,False
MAC000270,3585.75,15.70,62.15,22.14,False
MAC000306,1154.51,14.65,60.45,24.91,False
MAC000314,1047.64,16.92,72.66,10.42,False


Customer feature rows: 248
Customers with zero total design-period consumption: 0


In [3]:
feature_columns = [
    "design_kwh",
    "peak_share_pct",
    "shoulder_share_pct",
    "off_peak_share_pct",
]

share_columns = [
    "peak_share_pct",
    "shoulder_share_pct",
    "off_peak_share_pct",
]

assert customer_features[feature_columns].notna().all().all()

assert (
    customer_features[share_columns]
    .sum(axis=1)
    .sub(100)
    .abs()
    .lt(0.000001)
    .all()
)

feature_summary = (
    customer_features[feature_columns]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
    .T
)

print("Design-period customer feature distribution:")
display(feature_summary.round(2))

print("\nCustomers with the highest off-peak shares:")
display(
    customer_features.sort_values(
        "off_peak_share_pct",
        ascending=False,
    ).head(10).round(2)
)

feature_summary.to_csv(
    processed_folder / "customer_design_feature_summary_2013.csv"
)

Design-period customer feature distribution:


,count,mean,std,min,10%,25%,50%,75%,90%,max
design_kwh,248.0,2214.51,1779.35,442.35,805.03,1100.44,1776.62,2728.57,4127.18,18045.04
peak_share_pct,248.0,16.36,4.50,3.03,11.96,13.84,16.34,18.63,21.07,36.03
shoulder_share_pct,248.0,64.77,8.96,17.19,57.17,61.97,65.60,69.70,72.68,82.00
off_peak_share_pct,248.0,18.88,11.15,1.87,9.23,13.12,17.19,21.62,26.63,79.78



Customers with the highest off-peak shares:


,design_kwh,peak_share_pct,shoulder_share_pct,off_peak_share_pct,review_flag
household_id,,,,,
MAC004954,4587.77,3.03,17.19,79.78,False
MAC005263,4395.78,5.12,20.94,73.94,True
MAC004897,3890.44,5.11,23.17,71.73,False
MAC000331,3995.06,5.60,23.70,70.70,False
MAC000415,3558.94,6.93,25.05,68.02,False
MAC005041,7003.87,4.61,28.24,67.15,False
MAC004515,3969.12,9.70,33.93,56.37,False
MAC003465,777.80,4.33,43.94,51.73,False
MAC005309,4113.08,11.84,41.61,46.55,False


In [4]:
off_peak_threshold = customer_features["off_peak_share_pct"].quantile(0.75)
peak_threshold = customer_features["peak_share_pct"].quantile(0.75)

customer_features["usage_segment"] = "Other profiles"

customer_features.loc[
    customer_features["peak_share_pct"].ge(peak_threshold),
    "usage_segment",
] = "Higher peak share"

customer_features.loc[
    customer_features["off_peak_share_pct"].ge(off_peak_threshold),
    "usage_segment",
] = "Higher off-peak share"

segment_summary = (
    customer_features.groupby("usage_segment")
    .agg(
        households=("design_kwh", "size"),
        total_design_kwh=("design_kwh", "sum"),
        median_design_kwh=("design_kwh", "median"),
        median_peak_share_pct=("peak_share_pct", "median"),
        median_off_peak_share_pct=("off_peak_share_pct", "median"),
        flagged_households=("review_flag", "sum"),
    )
)

assert segment_summary["households"].sum() == 248

print("Off-peak threshold (%):", round(off_peak_threshold, 2))
print("Peak threshold (%):", round(peak_threshold, 2))
print("Higher off-peak share takes priority if both thresholds are met.")

display(segment_summary.round(2))

customer_features.to_csv(
    processed_folder / "customer_design_features_2013.csv"
)

segment_summary.to_csv(
    processed_folder / "customer_design_segment_summary_2013.csv"
)

Off-peak threshold (%): 21.62
Peak threshold (%): 18.63
Higher off-peak share takes priority if both thresholds are met.


,households,total_design_kwh,median_design_kwh,median_peak_share_pct,median_off_peak_share_pct,flagged_households
usage_segment,,,,,,
Higher off-peak share,62,169605.66,1912.88,13.25,25.14,2
Higher peak share,61,115264.43,1721.88,20.74,13.96,1
Other profiles,125,264328.89,1771.04,16.14,15.98,1


In [5]:
reference_rates = pd.Series({
    "Peak": 0.24,
    "Shoulder": 0.13,
    "Off-peak": 0.06,
})

customer_impact = customer_features.copy()

customer_impact["baseline_charge_gbp"] = (
    customer_impact["design_kwh"] * 0.14228
)

customer_impact["reference_charge_gbp"] = (
    customer_band_kwh.mul(
        reference_rates,
        axis="columns",
    ).sum(axis=1)
)

customer_impact["saving_gbp"] = (
    customer_impact["baseline_charge_gbp"]
    - customer_impact["reference_charge_gbp"]
)

customer_impact["saving_pct"] = (
    customer_impact["saving_gbp"]
    / customer_impact["baseline_charge_gbp"]
    * 100
)

customer_impact["pays_more"] = (
    customer_impact["saving_gbp"].lt(-1e-9)
)

segment_impact = customer_impact.groupby("usage_segment").agg(
    households=("design_kwh", "size"),
    median_saving_pct=("saving_pct", "median"),
    total_saving_gbp=("saving_gbp", "sum"),
    households_paying_more=("pays_more", "sum"),
)

segment_impact["paying_more_pct"] = (
    segment_impact["households_paying_more"]
    / segment_impact["households"] * 100
)

display(segment_impact.round(2))

customer_impact.to_csv(
    processed_folder / "customer_design_reference_tariff_impact.csv"
)

segment_impact.to_csv(
    processed_folder / "segment_design_reference_tariff_impact.csv"
)

,households,median_saving_pct,total_saving_gbp,households_paying_more,paying_more_pct
usage_segment,,,,,
Higher off-peak share,62,10.75,3969.16,0,0.00
Higher peak share,61,-0.39,-273.18,39,63.93
Other profiles,125,4.63,1536.14,6,4.80


In [6]:
import json

segment_rules = {
    "design_period": {
        "start_utc": "2013-01-01T00:00:00Z",
        "end_exclusive_utc": "2013-07-01T00:00:00Z",
    },
    "off_peak_threshold_pct": float(off_peak_threshold),
    "peak_threshold_pct": float(peak_threshold),
    "priority": [
        "Higher off-peak share",
        "Higher peak share",
        "Other profiles",
    ],
    "reference_rates_gbp_per_kwh": reference_rates.to_dict(),
    "limitations": [
        "Segments describe usage patterns, not willingness to change usage.",
        "Reference rates were selected in a six-household pilot.",
        "Bill comparisons exclude standing charges and behavioural changes.",
        "Review flags use full-year data and are for sensitivity analysis only.",
    ],
}

rules_file = processed_folder / "customer_segment_rules_2013.json"

rules_file.write_text(
    json.dumps(segment_rules, indent=2),
    encoding="utf-8",
)

connection.close()

print("Segment rules saved:", rules_file)
print("Database connection closed.")

Segment rules saved: D:\electricity-tariff-optimisation\data\processed\customer_segment_rules_2013.json
Database connection closed.
